In [29]:
import pandas as pd
import getpass
import yaml
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
import sys

sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.pubchem_retrieval as pcr
import metatlas2.load_tools as lt

In [30]:
# Required: Input file with compounds to add. Minimally, must have columns 'inchi_key' and 'label'
COMPOUNDS_INPUT = "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv"

In [ ]:
# Load configuration from YAML file
config_path = Path("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

METATLAS_DB_PATH = config["paths"]["main_database"]
PUBCHEM_CACHE_PATH = config["paths"]["pubchem_cache"]
FORCE_CACHE_UPDATE = config["database_options"]["force_cache_update"]
ADD_COMPOUND_DUPLICATES = config["database_options"]["add_compound_duplicates"]
ANALYST_NAME = getpass.getuser()
TIMESTAMP = datetime.now().isoformat()

## Load and Validate Input Data

In [32]:
compounds_df = lt.load_compound_input(COMPOUNDS_INPUT)

Loading input data from: /Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv
Loaded 373 rows from input table


## PubChem Data Retrieval Functions

## Retrieve PubChem Data for All Compounds

In [33]:
pcr.retrieve_pubchem_info(compounds_df, 
    PUBCHEM_CACHE_PATH, 
    TIMESTAMP, 
    FORCE_CACHE_UPDATE)

Loaded global PubChem cache with 482 entries from /Users/BKieft/Metabolomics/metatlas2/data/databases/pubchem_cache/pubchem_global_cache.pkl
Compounds already in cache: 333
Compounds needing PubChem lookup: 0
All compounds already in cache!

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.

PubChem data retrieval complete!
    Total compounds in global cache: 482
    Successful PubChem retrievals in cache: 478
    Failed retrievals in cache: 4


## Add Compounds to Database

In [ ]:
dbi.add_compounds_to_db(
    compounds_df, 
    PUBCHEM_CACHE_PATH, 
    METATLAS_DB_PATH, 
    ANALYST_NAME,
    COMPOUNDS_INPUT,
    ADD_COMPOUND_DUPLICATES
)

Adding 333 compounds to database: /Users/BKieft/Metabolomics/metatlas2/data/databases/metatlas.duckdb
Loaded global PubChem cache with 482 entries from /Users/BKieft/Metabolomics/metatlas2/data/databases/pubchem_cache/pubchem_global_cache.pkl
Found 443 existing compounds in database. Not creating duplicates.


Processing compounds:   0%|          | 0/373 [00:00<?, ?it/s]


Compounds added successfully!
   Total compounds in database: 482
   New compounds created: 39
   Compounds skipped (already existed): 334
   Total RT/MZ references in database: 860
   New RT/MZ references created: 339
   RT/MZ references skipped (duplicates): 34


In [ ]:
dbi.validate_database(METATLAS_DB_PATH)


Database Validation:
   Compounds: 482
   RT/MZ References: 860
   Atlases: 0
   Atlas-Compound associations: 0
   Method combinations:
      HILICZ/negative: 456 references
      HILICZ/positive: 404 references
